# Aperture

Aperture is a SAT-based optimization tool as well as an anytime MaxSAT solver. 
In addition to the native C++ API, `Aperture` also provides a Python API for users who prefer to work in Python. 
The Python API (`PyPerture`) allows users to easily integrate `Aperture` into their Python workflows and take advantage of its solving capabilities.
In this notebook we will be demonstrating how to use `PyAperture`.

# Installation

First, we need to install nanobind in order to compile `Aperture` as a Python module.

```bash
pip install nanobind
```

Next, we need to compile the `PyPerture` module. In the root directory run:

```bash
make lpy
```

This will create a `PyAperture` module in the python directory that we can import in our Python code.
Note: you may need to adjust the `Makefile` if you have a different setup or if you want to specify a different Python version.
Since the python interpreter will first look for the module in the current directory, you can simply copy the `PyAperture` module to your working directory or add the path to the module to your `PYTHONPATH` environment variable.

# Usage

Once you have the `PyAperture` module installed, you can import it in your Python as `pyperture`.

```python
import pyperture as pa
```

## Core API

The core API of `PyAperture` consists of simple functions that allow you to create a solver instance and add clauses to it.
The main class is the solver itself (`pyperture.Solver`) which provides methods for adding clauses and solving the problem.
Here is a simple example of how to use the core API:

In [1]:
import pyperture as pa

# Create a solver instance (with Glucose 4.1 as the underlying SAT solver)

solver = pa.Solver(sat_solver="glucose")

# Create variables

x1 = solver.new_var()  # Create variable x1
x2 = solver.new_var()  # Create variable x2

# Add clauses to the solver

c1 = pa.lits([x1, x2])  # Create clause (x1 OR x2)
c2 = pa.lits([-x1, x2])  # Create clause (NOT x1 OR x2)

solver.add_clause(c1)  # Add clause (x1 OR x2) to the solver
solver.add_clause(c2)  # Add clause (NOT x1 OR x2) to the solver

c Using Glucose as main SAT solver.


True

In this example, we create a solver instance, add two clauses to it.

Note that every variable used must be created using the `new_var` method of the solver instance.

Also, when passing lists of literals to any function, it is recommended to wrap them with the `lits` type. Doing so will prevent further copying of the list when passed to the underlying C++ code, which can improve performance. In addition, there are functions with the same name that differ only in the type of the arguments they accept, so in order to use them with different types of arguments (e.g. `wlits`) in the same context, it is necessary to wrap the lists with the `lits` type.

There are several other core functions available, as well as some helper functions for working with the solver.

# Solving

After adding clauses to the solver, you can call the `solve` method to find a solution to the problem. That is, `solve` will solve SAT under assumptions for the added clauses. Here is an example of how to use the `solve` method:

In [2]:

# Solve the problem

sat = solver.solve()

if sat:
    print("SAT")
    print("Model:", solver.get_latest_solution())
else:
    status = solver.get_latest_status()
    if status == "UNSAT":
        print("UNSAT")
    elif status == "ERROR":
        print("ERROR: ", solver.get_latest_error_message())
    else:
        print("UNKNOWN")

SAT
Model: pyperture.lits([-1, 2])


That is, the code above determines if the following formula is satisfiable:
$(x1 \lor x2) \land (\lnot x1 \lor x2)$

As mentioned above, every solving can be done under hard assumptions, i.e. you can specify a list of literals that must be true in the solution. For example, if we want to solve the problem under the assumption that $x_1$ is true, we can do:

```python
sat = solver.solve(assumptions=[x1])
```

And under the assumption that $x_1$ is false, we can do:

```python
sat = solver.solve(assumptions=[-x1])
```

## Solving MaxSAT

Given a propositional formula $\varphi$ in CNF, over the boolean variables $X=\{x_1,x_2,...,x_n\}$, and a set of soft literals $S=\{s_1,s_2,...,s_m\}$, where each $s_j$ is either a variable or its negation. In addition, each soft literal $s_j,j=1,...,m$ is assigned a weight $w_j > 0$. The MaxSAT problem is to find an assignment $\mu \in \{0,1\}^n$ to the variables in $X$ that satisfies $\varphi$ and minimizes the total weight of unsatisfied soft literals in $S$.

Formally, we can define the MaxSAT problem as follows:

$$
\begin{aligned}
\underset{\mu}{\text{minimize}} \quad & \sum_{j=1}^{m} w_j \cdot \mu(s_j) \\
\text{subject to} \quad & \varphi \\
& x_{i}\in \{0, 1\}, \quad i=1,...,n
\end{aligned}
$$

`Aperture` allows anytime MaxSAT solving through two API query functions, `solve_maxsat` for Unweighted MaxSAT (i.e. all weights are 1) and `solve_weighted_maxsat` for Weighted MaxSAT.

For example, let $X=\{x_1,x_2\}, S=\{x_1,x_2\}, \varphi=(x_1 \lor x_2) \land (\lnot x_1 \lor x_2)$, and we want to solve Unweighted MaxSAT problem with soft literals.

In [3]:
# Create a solver instance (with Glucose 4.1 as the underlying SAT solver)

solver = pa.Solver(sat_solver="glucose")

solver.set_verbosity_level(0) # So we wont get logs flooding for this example

# Create variables

x1 = solver.new_var()
x2 = solver.new_var()

# Add clauses to the solver

solver.add_clause(pa.lits([x1, x2]))
solver.add_clause(pa.lits([-x1, x2]))

# Solve the Unweighted MaxSAT problem

soft_lits = pa.lits([x1, x2])  # Soft literals
assumptions = pa.lits([])  # No hard assumptions
sat = solver.solve_maxsat(assumptions=assumptions, soft_lits=soft_lits, fix_model_value=True)

if sat:
    print("Model:", solver.get_latest_solution())
    if solver.is_latest_maxsat_optimal():
        print("Optimal solution found.")
    else:
        print("Satisfabile.")
    print("Objective value:", solver.get_latest_maxsat_value())
else:
    status = solver.get_latest_status()
    if status == "UNSAT":
        print("UNSAT")
    elif status == "ERROR":
        print("ERROR: ", solver.get_latest_error_message())
    else:
        print("UNKNOWN")

c Using Glucose as main SAT solver.
Model: pyperture.lits([-1, 2])
Optimal solution found.
Objective value: 1


Note the `fix_model_value` parameter in the `solve_maxsat` function, which allows the solver to fix the value of the model variables to the values of the solution found (this will be carried on to next calls to the solver). It is useful for performance reasons, as it allows the solver to bound the value of the current best solution found in its Complete Part using clauses, instead of assumptions, which can be more efficient in some cases. Since as menssioned, the fixing is only done if `fix_model_value` is set to `True` and if the solver was able to do it in its Complete Part, you can check if the fixing was done by calling the `is_latest_maxsat_fixed_model_value` function, which returns `True` if the solver was able to fix the model variables and `False` otherwise.

The same can be done for Weighted MaxSAT by using the `solve_weighted_maxsat` function and providing weights for the soft literals. For example if we want to solve a Weighted MaxSAT problem with the previous settings, with weights $w_{1}=1$ and $w_{2}=2$.

In [4]:
# Create a solver instance (with Glucose 4.1 as the underlying SAT solver)

solver = pa.Solver(sat_solver="glucose")

solver.set_verbosity_level(0) # So we wont get logs flooding for this example

# Create variables

x1 = solver.new_var()
x2 = solver.new_var()

# Add clauses to the solver

solver.add_clause(pa.lits([x1, x2]))
solver.add_clause(pa.lits([-x1, x2]))

# Solve the Unweighted MaxSAT problem

soft_wlits = pa.wlits([(1, x1), (2, x2)])  # Weighted soft literals, note the (weight, literal) format
assumptions = pa.lits([])  # No hard assumptions
sat = solver.solve_weighted_maxsat(assumptions=assumptions, soft_wlits=soft_wlits, fix_model_value=True)

if sat:
    print("Model:", solver.get_latest_solution())
    if solver.is_latest_maxsat_optimal():
        print("Optimal solution found.")
    else:
        print("Satisfabile.")
    print("Objective value:", solver.get_latest_maxsat_value())
else:
    status = solver.get_latest_status()
    if status == "UNSAT":
        print("UNSAT")
    elif status == "ERROR":
        print("ERROR: ", solver.get_latest_error_message())
    else:
        print("UNKNOWN")

c Using Glucose as main SAT solver.
Model: pyperture.lits([-1, 2])
Optimal solution found.
Objective value: 2


## Solving Black-Box Optimization

Let $\varphi$ be a propositional formula in CNF, over the boolean variables $X=\{x_1,x_2,...,x_n\}$, and a set of observable literals $O=\{o_1,o_2,...,o_k\}$, where each $o_j$ is either a boolean variable or its negation. In addition, we have an objective function $\Psi : \{0,1\}^k \mapsto \mathbb{R}_{\geq 0}$, strictly monotone in the observable literals, that is, for every two assignments $\mu_1$ and $\mu_2$ such that $\mu_1(o_j) \leq \mu_2(o_j)$ for every $o_j \in O$, we have $\Psi(\mu_1) \leq \Psi(\mu_2)$. The Black-Box Optimization Problem is to find an assignment $\mu \in \{0,1\}^n$ to the variables in $X$ that satisfies $\varphi$ and minimizes the value of the objective function $\Psi$ when applied to the observable literals in $O$.
Formally, we can define the problem as follows:

$$
\begin{aligned}
\underset{\mu}{\text{minimize}} \quad & \Psi(\mu(o_1), \mu(o_2), ..., \mu(o_k)) \\
\text{subject to} \quad & \varphi \\
& x_{i}\in \{0, 1\}, \quad i=1,...,n
\end{aligned}
$$

`Aperture` allows solving black-box optimization problems through the `solve_black_box` function, which takes as input a black-box objective function and a list of observables to optimize over. Naturally it is incomplete, but it can be useful in cases where the objective function is complex and we may not have a good understanding of its structure, but we can still evaluate it for different assignments.
For example, Let: 
- $X=\{x_1,x_2,x_3\}$
- $O=\{x_1,x_2\}$
- $\varphi = (x_1 \lor x_2) \land (\lnot x_1 \lor x_2) \land (x_3)$
- $\Psi(o_1,o_2)=2 \cdot o_1 + 3 \cdot o_2 + o_1 \cdot o_2$

In [5]:
# Create a solver instance (with Glucose 4.1 as the underlying SAT solver)

solver = pa.Solver(sat_solver="glucose")

solver.set_verbosity_level(0) # So we wont get logs flooding for this example

# Create variables

x1 = solver.new_var()
x2 = solver.new_var()
x3 = solver.new_var()

# Add clauses to the solver

solver.add_clause(pa.lits([x1, x2]))
solver.add_clause(pa.lits([-x1, x2]))
solver.add_clause(pa.lits([x3]))

# Define the black-box objective function

def black_box_objective(lit_val_func):
    # Get the values of x1 and x2 from the literal value function
    o1 = lit_val_func(x1) == x1 # Convert literal value to boolean
    o2 = lit_val_func(x2) == x2
    # Compute the objective value
    return 2 * o1 + 3 * o2 + o1 * o2

# Solve the black-box optimization problem

assumptions = pa.lits([])  # No hard assumptions
observables = pa.lits([x1, x2])  # o_1 = x1, o_2 = x2
sat = solver.solve_black_box(assumptions=assumptions, observables=observables, pb_func=black_box_objective)

if sat:
    print("SAT")
    print("Model:", solver.get_latest_solution())
    print("Objective value:", solver.get_latest_black_box_value())
else:
    status = solver.get_latest_status()
    if status == "UNSAT":
        print("UNSAT")
    elif status == "ERROR":
        print("ERROR: ", solver.get_latest_error_message())
    else:
        print("UNKNOWN")

SAT
Model: pyperture.lits([-1, 2, 3])
Objective value: 3.0
c Using Glucose as main SAT solver.


## Solving Modulo Bit-Vector Optimization (OBV)

Let $\varphi$ be a propositional formula in CNF, over the boolean variables $X=\{x_1,x_2,...,x_n\}$, and a set of target bits $T=\{t_{m-1},t_{m-2},...,t_0\}$, where each $t_j$ is either a variable or its negation. The Modulo Bit-Vector Optimization (OBV) problem is to find an assignment $\mu \in \{0,1\}^n$ to the variables in $X$ that satisfies $\varphi$ and minimizes the value of the target bit vector $T$, interpreted as a binary number ($t_{m}$ is the Most Significant Bit - MSB).
Formally, we can define the problem as follows:

$$
\begin{aligned}
\underset{\mu}{\text{minimize}} \quad & \sum_{j=0}^{m-1} \mu(t_j) \cdot 2^j \\
\text{subject to} \quad & \varphi \\
& x_{i}\in \{0, 1\}, \quad i=1,...,n
\end{aligned}
$$

`Aperture` allows solving OBV problems through the `solve_obv` function, which takes as input a list of bits representing the target and a list of hard assumptions. For example, Let $X=\{x_1,x_2,x_3\}, T=\{x_1,x_2\}, \varphi = (x_1 \lor x_2) \land (\lnot x_1 \lor x_2) \land (x_3)$.

In [6]:
# Create a solver instance (with Glucose 4.1 as the underlying SAT solver)

solver = pa.Solver(sat_solver="glucose")

solver.set_verbosity_level(0) # So we wont get logs flooding for this example

# Create variables

x1 = solver.new_var()
x2 = solver.new_var()
x3 = solver.new_var()

# Add clauses to the solver

solver.add_clause(pa.lits([x1, x2]))
solver.add_clause(pa.lits([-x1, x2]))
solver.add_clause(pa.lits([x3]))

# Define the target and the corresponding bits

assumptions = pa.lits([])  # No hard assumptions
targets = pa.lits([x1, x2])  # t1 = x1, t0 = x2

# Solve the OBV problem

sat = solver.solve_obv(assumptions=assumptions, targets=targets)

if sat:
    print("SAT")
    print("Model:", solver.get_latest_solution())
else:
    status = solver.get_latest_status()
    if status == "UNSAT":
        print("UNSAT")
    elif status == "ERROR":
        print("ERROR: ", solver.get_latest_error_message())
    else:
        print("UNKNOWN")

SATc Using Glucose as main SAT solver.

Model: pyperture.lits([-1, 2, 3])


## Additional Parameter for Query Functions

For the query functions (e.g. `solve`, `solve_maxsat`, `solve_black_box`...) there is an additional parameter called `callback_on_solution_found`, which allows the user to specify a callback function that will be called every time a new (improving) solution is found. The callback function should take the list of literals being optimized over as a parameter and return a boolean value - True if the solver should stop, and False if the solver should continue solving.

In [7]:
import pyperture as pa

solver = pa.Solver(sat_solver="glucose")

x1 = solver.new_var()
x2 = solver.new_var()

def solution_callback(lits):
    cost = 0
    for lit in lits:
        cost += solver.lit_value(lit) == lit
    return cost <= 1

assumps = pa.lits([x1])
soft_lits = pa.lits([x1, x2])

# Stops right after the initial solution, since its value is 1:
sat = solver.solve_maxsat(assumps, soft_lits, False, solution_callback)

# Continues with the search in order to prove that 1 is the optimal value:
sat = solver.solve_maxsat(assumps, soft_lits, False)

if sat:
    print("Model:", solver.get_latest_solution())
    if solver.is_latest_maxsat_optimal():
        print("Optimal solution found.")
    else:
        print("Satisfabile.")
    print("Objective value:", solver.get_latest_maxsat_value())
else:
    status = solver.get_latest_status()
    if status == "UNSAT":
        print("UNSAT")
    elif status == "ERROR":
        print("ERROR: ", solver.get_latest_error_message())
    else:
        print("UNKNOWN")

Model:c Using Glucose as main SAT solver.
 pyperture.lits([1, -2])
Optimal solution found.
Objective value: 1
o 1
c timeo 0 1
o 1
c timeo 0 1
c build NuWeighting instance start!
c nuweighting_nvars = 2
c nuweighting_nclauses = 1
c nuweighting_topclauseweight = 3
c problem_weighted = 0
c unsat_assumps_soft_weight = 1
c before build neighbor
c build instime is 0
c using neighbor
c build NuWeighting instance done!
c changing to NuWeighting solver!!!
c problem weighted = 0
c nuweightingTimeLimit = 15
c nuweighting search done!
c step 1 get_runtime 0 time_limit_for_ls 15
c Starting MRS-Beaver with max iterations 2147483647 and obv conflict threshold 1000.
c MRS-Beaver stopping due to size.
c Starting LSU with encoder type 0.
c LSU finished with optimal value: 1.


# Constraints

In addition to clauses, `Aperture` also allows adding Cardinality and Pseudo-Boolean Constraints to the solver. \
Let $X=\{x_1,x_2,...,x_n\}$ be a set of boolean variables and $L=\{l_1,l_2,...,l_m\}$ be a set of literals (a boolean variable or its negation) over $X$. A Cardinality Constraint is a constraint of the form $\sum_{j=1}^{m} l_j \oplus k$ for some $k\in \mathbb{N}$ where $\oplus \in \{\le, \leq, =, \geq, \gt\}$. \
A Pseudo-Boolean Constraint is a generalization of Cardinality Constraint, where each literal $l_j\in L$ has a coefficient $w_j \in \mathbb{N}$ i.e. $\sum_{j=1}^{m} w_j \cdot l_j \oplus k$ for some some $k\in \mathbb{N}$ where $\oplus \in \{\le, \leq, =, \geq, \gt\}$. \
`Aperture` allows adding these constraints to the solver using API functions with a respective name according to the constraint type.

---
**NOTE**

For Pseudo-Boolean constraints, `Aperture` currently only supports the $\lt$ and $\leq$ predicates. For Cardinality constraints, all predicates are supported.

---

For example, Let $X=\{x_1,x_2,x_3\}$ and we want to add the cardinality constraint $x_1 + x_2 + \lnot x_3 \leq 2$, we can do:

In [8]:
import pyperture as pa

solver = pa.Solver(sat_solver="glucose")

x1 = solver.new_var()
x2 = solver.new_var()
x3 = solver.new_var()

soft_lits = pa.lits([x1, x2, -x3])

solver.add_constraint_less_than_equal(soft_lits, 2)

solver.solve()

print("Model:", solver.get_latest_solution())

Model: pyperture.lits([-1, -2, 3])
c Using Glucose as main SAT solver.


For the Pseudo-Boolean constraint $3\cdot x_1 + 2 \cdot x_2 + 4 \cdot \lnot x_3 \leq 2$, we can do:

In [9]:
import pyperture as pa

solver = pa.Solver(sat_solver="glucose")

x1 = solver.new_var()
x2 = solver.new_var()
x3 = solver.new_var()

soft_wlits = pa.wlits([(3, x1), (2, x2), (4, -x3)])

solver.add_constraint_less_than_equal(soft_wlits, 2)

solver.solve()

print("Model:", solver.get_latest_solution())

Model:c Using Glucose as main SAT solver.
 pyperture.lits([-1, -2, 3])


Note that the constraints also return a boolean value indicating whether the constraint was added successfully or not. The caluses and variables used in the encoding will not be added to the solver if the return value is False.

In order to allow activation and deactivation of the constraints, you can also provide a selector literal to the constraint functions. The selector literal will be added to the clauses encoding the constraint, and it will allow you to activate or deactivate the constraint by setting the value of the selector literal in the assumptions of the solver. For example, if we want to add the cardinality constraint $x_1 + x_2 + \lnot x_3 \leq 2$ with a selector literal $s$, we can do:

In [10]:
import pyperture as pa

solver = pa.Solver(sat_solver="glucose")

x1 = solver.new_var()
x2 = solver.new_var()
x3 = solver.new_var()
s = solver.new_var()  # Selector literal

soft_lits = pa.lits([x1, x2, -x3])

solver.add_constraint_greater_than_equal(soft_lits, 2, selector=s)

# To activate the constraint, we set the selector literal to false in the assumptions:

solver.solve_maxsat(assumptions=pa.lits([-s]), soft_lits=soft_lits, fix_model_value=False)

print("Model with constraint activated:", solver.get_latest_solution())

# To deactivate the constraint, we set the selector literal to true in the assumptions:

solver.solve_maxsat(assumptions=pa.lits([s]), soft_lits=soft_lits, fix_model_value=False)

print("Model with constraint deactivated:", solver.get_latest_solution())

Model with constraint activated: pyperture.lits([-1, 2, -3, -4])
Model with constraint deactivated: pyperture.lits([-1, -2, 3, 4])
c Using Glucose as main SAT solver.
o 2
c timeo 0 2
c build NuWeighting instance start!
c nuweighting_nvars = 9
c nuweighting_nclauses = 20
c nuweighting_topclauseweight = 4
c problem_weighted = 0
c unsat_assumps_soft_weight = 0
c before build neighbor
c build instime is 0
c using neighbor
c build NuWeighting instance done!
c changing to NuWeighting solver!!!
c problem weighted = 0
c nuweightingTimeLimit = 15
c nuweighting search done!
c step 10000000 get_runtime 0.74 time_limit_for_ls 15
c Starting MRS-Beaver with max iterations 2147483647 and obv conflict threshold 1000.
c MRS-Beaver stopping due to size.
c Starting LSU with encoder type 0.
c LSU finished with optimal value: 2.
o 0
c timeo 0 0


If you want to permanently activate / deactivate the constraint, you can simply add a unit clause with the selector literal or its negation to the solver. For this specific purpose, doing so might result in a more efficient solving.

## Totalizer Encoding

The Totalizer encoding is a specific encoding for cardinality constraints to CNF. It is used as the default encoding for cardinality constraints in `Aperture`, and it is also used for encoding the cardinality constraints that are generated internally by `Aperture` for solving MaxSAT problems.
The Totalizer encoding returns a list of auxiliary variables that encodes the unary sum of the input literals, which can be used for adding additional constraints on the sum of the literals.

In general, the encoding requires $O(m \log m)$ auxiliary variables and $O(m^2)$ clauses, where $m$ is the number of literals in the input.

For example, if we want to encode the cardinality constraint $x_1 + x_2 + \lnot x_3 \leq 2$ using the Totalizer encoding, the output of the totalizer will be $T=\{t_1, t_2, t_3\}$, where $t_j$ is true if at least $j$ literals in the input are true. Therefore, the constraint $x_1 + x_2 + \lnot x_3 \leq 2$ can be encoded as $\neg t_3$, which means that at most 2 literals can be true. In addition, if we want to add the constraint $x_1 + x_2 + \lnot x_3 \leq 1$, we can encode it as $\neg t_2$, which means that at most 1 literal can be true.

Note that the totalizer encoding requires a selector literal in order to be able to activate and deactivate the constraint.

We can also use the encoding for other predicates like $x_1 + x_2 + \lnot x_3 \ge 1$, which can be encoded as $t_1$, meaning that at least 1 literal must be true. For the equality predicate, we can encode $x_1 + x_2 + \lnot x_3 = 2$ as $\neg t_3 \land t_2$, which means that at most 2 literals can be true and at least 2 literals must be true, which is equivalent to saying that exactly 2 literals must be true.

In [11]:
import pyperture as pa

solver = pa.Solver(sat_solver="glucose")

x1 = solver.new_var()
x2 = solver.new_var()
x3 = solver.new_var()
s = solver.new_var()  # Selector literal

soft_lits = pa.lits([x1, x2, -x3])

totalizer = solver.get_totalizer(soft_lits, selector=s)

print("Totalizer variables:", totalizer)

assumptions = pa.lits([-s])  # Activate the totalizer constraint

# At most 2 literals can be true

assumptions.append(-totalizer[2])  
solver.solve_maxsat(assumptions=assumptions, soft_lits=soft_lits, fix_model_value=False)
print("At most 2 literals can be true, Model:", solver.get_latest_solution())

assumptions.pop()  # Remove the previous constraint

# At least 1 literal must be true

assumptions.append(totalizer[0])
solver.solve_maxsat(assumptions=assumptions, soft_lits=soft_lits, fix_model_value=False)
print("At least 1 literal must be true, Model:", solver.get_latest_solution())

assumptions.pop()  # Remove the previous constraint

# Exactly 2 literals must be true

assumptions.append(totalizer[1])  # At least 2 literals must be true
assumptions.append(-totalizer[2])  # At most 2 literals can be true
solver.solve_maxsat(assumptions=assumptions, soft_lits=soft_lits, fix_model_value=False)
print("Exactly 2 literals must be true, Model:", solver.get_latest_solution())

Totalizer variables:c Using Glucose as main SAT solver.
 pyperture.lits([7, 8, 9])
At most 2 literals can be true, Model: pyperture.lits([-1, -2, 3, -4])
o 0
c timeo 0 0
o 1
c timeo 0 1
c build NuWeighting instance start!
c nuweighting_nvars = 9
c nuweighting_nclauses = 17
c nuweighting_topclauseweight = 4
c problem_weighted = 0
c unsat_assumps_soft_weight = 0
c before build neighbor
c build instime is 0
c using neighbor
c build NuWeighting instance done!
c changing to NuWeighting solver!!!
c problem weighted = 0
c nuweightingTimeLimit = 15
c nuweighting search done!
c step 10000000 get_runtime 1.22 time_limit_for_ls 15
c Starting MRS-Beaver with max iterations 2147483647 and obv conflict threshold 1000.
c MRS-Beaver stopping due to size.
c Starting LSU with encoder type 0.
cAt least 1 literal must be true, Model: pyperture.lits([-1, -2, -3, -4])
Exactly 2 literals must be true, Model: pyperture.lits([-1, 2, -3, -4])
 LSU finished with optimal value: 1.
o 2
c timeo 1 2
c build NuWeight

### Right Hand Side Simplification

For $\le$ and $\leq$ constraints,when the right hand side of the cardinality constraint is less than the number of literals in the input, we can simplify the constraint by removing the literals that are above the right hand side (+1) and thus reducing the amount of auxiliary variables and clauses generated by the encoding.

In general, it reduces the number of clauses that the encoding will generate to $O(k \cdot m)$ where $k$ is the right hand side of the constraint and $m$ is the number of literals in the input, which can be a significant improvement if $k \ll m$.

For example, if we want to encode the cardinality constraint $x_1 + x_2 + \lnot x_3 \leq 1$, the simplified output of the totalizer will be $T=\{t_1, t_2\}$ where $t_2$ is true if at least 2 literals in the input are true.

To enable this simplification, you can simply set the `rhs_simplification` (which is optional) parameter of the totalizer function to the right hand side of the constraint.

In [12]:
import pyperture as pa

solver = pa.Solver(sat_solver="glucose")

x1 = solver.new_var()
x2 = solver.new_var()
x3 = solver.new_var()
s = solver.new_var()  # Selector literal

c1 = pa.lits([x1, x2])
c2 = pa.lits([-x1, x2])

solver.add_clause(c1)
solver.add_clause(c2)

soft_lits = pa.lits([x1, x2, -x3])
k = 1  # Right hand side of the constraint

totalizer = solver.get_totalizer(soft_lits, selector=s, rhs_simplification=k)

print("Totalizer variables - simplified:", totalizer)

assumptions = pa.lits([-s])  # Activate the totalizer constraint
assumptions.append(-totalizer[k])  # At most k=1 literals can be true

solver.solve_maxsat(assumptions=assumptions, soft_lits=soft_lits, fix_model_value=False)
print("At most 1 literal can be true, Model:", solver.get_latest_solution())

Totalizer variables - simplified: pyperture.lits([7, 8])
c Using Glucose as main SAT solver.
At most 1 literal can be true, Model: pyperture.lits([-1, 2, 3, -4])
o 1
c timeo 0 1
c build NuWeighting instance start!
c nuweighting_nvars = 8
c nuweighting_nclauses = 16
c nuweighting_topclauseweight = 4
c problem_weighted = 0
c unsat_assumps_soft_weight = 0
c before build neighbor
c build instime is 0
c using neighbor
c build NuWeighting instance done!
c changing to NuWeighting solver!!!
c problem weighted = 0
c nuweightingTimeLimit = 15
c nuweighting search done!
c step 10000000 get_runtime 0.66 time_limit_for_ls 15
c Starting MRS-Beaver with max iterations 2147483647 and obv conflict threshold 1000.
c MRS-Beaver stopping due to size.
c Starting LSU with encoder type 0.
c LSU finished with optimal value: 1.


## Generalized Totalizer Encoding

The Generalized Totalizer encoding is a generalization of the Totalizer encoding for Pseudo-Boolean constraints. It is used as the default encoding for Pseudo-Boolean constraints in `Aperture`, and it is also used for encoding the Pseudo-Boolean constraints that are generated internally by `Aperture` for solving MaxSAT problems.

The Generalized Totalizer encoding returns a list of pairs $(w_j, t_j)$ where $w_j$ is a weight and $t_j$ is an auxiliary variables that encodes the sum $w_j$ of the input literals, which can be used for adding additional constraints on the sum of the literals.

Note that the encoding supports only the $\leq$ and $\lt$ predicates and it requires inserting *all* $(*)$ the literals above the desired bound as assumptions or unit clauses in order to be able to encode the constraint.

For example, if we want to encode the Pseudo-Boolean constraint $3 \cdot x_1 + 2 \cdot x_2 + 4 \cdot \lnot x_3 \leq 5$, the output of the generalized totalizer will be $T=\{(2, t_1), (3, t_2), (4, t_3), (5, t_4), (6, t_5), (7, t_6), (9, t_7)\}$, where $t_j$ is true if at least $j$ weight in the input is true. Therefore, the constraint $3 \cdot x_1 + 2 \cdot x_2 + 4 \cdot \lnot x_3 \leq 5$ can be encoded as $\neg t_5 \wedge \neg t_6 \wedge \neg t_7$ $(*)$, which means that at most $w_4 = 5$ weight can be true.

In [13]:
import pyperture as pa

solver = pa.Solver(sat_solver="glucose")

x1 = solver.new_var()
x2 = solver.new_var()
x3 = solver.new_var()
s = solver.new_var()  # Selector literal

soft_wlits = pa.wlits([(3, x1), (2, x2), (4, -x3)])

totalizer = solver.get_gen_totalizer(soft_wlits, selector=s)

print("Totalizer output (weight, variable):", totalizer)

assumptions = pa.lits([-s])  # Activate the totalizer constraint
i = len(totalizer) - 1
while i >= 0 and totalizer[i][0] > 5:
    assumptions.append(-totalizer[i][1])
    i -= 1

solver.solve_weighted_maxsat(assumptions=assumptions, soft_wlits=soft_wlits, fix_model_value=False)

print("At most 5 weight can be true, Model:", solver.get_latest_solution())

Totalizer output (weight, variable):c Using Glucose as main SAT solver.
 pyperture.wlits([(2, 11), (3, 9), (4, 8), (5, 10), (6, 14), (7, 12), (9, 13)])
At most 5 weight can be true, Model: pyperture.lits([-1, -2, 3, -4])
o 0
c timeo 0 0


Right handside simplification can also be applied for the generalized totalizer encoding, similarly to the totalizer encoding, by setting the `rhs_simplification` parameter to the desired right hand side of the constraint.

In [14]:
import pyperture as pa

solver = pa.Solver(sat_solver="glucose")

x1 = solver.new_var()
x2 = solver.new_var()
x3 = solver.new_var()
s = solver.new_var()  # Selector literal

soft_wlits = pa.wlits([(3, x1), (2, x2), (4, -x3)])

totalizer = solver.get_gen_totalizer(soft_wlits, selector=s, rhs_simplification=5)

print("Totalizer output (weight, variable) - simplified:", totalizer)

assumptions = pa.lits([-s])  # Activate the totalizer constraint
i = len(totalizer) - 1
while i >= 0 and totalizer[i][0] > 5:
    assumptions.append(-totalizer[i][1])
    i -= 1

solver.solve_weighted_maxsat(assumptions=assumptions, soft_wlits=soft_wlits, fix_model_value=False)

print("At most 5 weight can be true, Model:", solver.get_latest_solution())

Totalizer output (weight, variable) - simplified:c Using Glucose as main SAT solver.
 pyperture.wlits([(2, 11), (3, 9), (4, 8), (5, 10), (6, 12)])
At most 5 weight can be true, Model: pyperture.lits([-1, -2, 3, -4])
o 0
c timeo 0 0
